In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score
import string
from nltk.corpus import stopwords
import re

df = pd.read_csv('fake-news-pt-eu/Final_dataset_portuguese.csv', header=0)
df = df.reset_index()  # make sure indexes pair with number of rows


In [3]:
#cleaning the data function ISTO DEMORA BUE


def clean_data(text,stops):
    text = text.lower()
    trans_table = str.maketrans({p: " " for p in string.punctuation})
    text = text.translate(trans_table)
    #precisamos de tirar dias da semana aqui-----------------------------------------------------------
    words = text.split()
    final = []
    for word in words:
         if word not in stops:
             final.append(word)
    final = " ".join(final)
    
    #forma exotica para remover pontuacao num texto
    #final = final.translate(str.maketrans("","",string.punctuation))
    
    # NOTA: caso os numeros nao tenham impacto na info dos textos
    # usamos isto para remover numeros (e talvez tenhamos de remover % tambem)
    # se os numeros forem importantes, talvez temos de modificar o texto de forma a indentificar os numeros
    # ex: 74,5% --> #PERCENT74,5
    
    final = "".join([i for i in final if not i.isdigit()])
    #forma NADA efeciente de tirar espaços
    while "  " in final:
        final = final.replace("  ", " ")
    return final
#the new data frame is going to have the same format but the text will be altered

base_weekdays = [
    "segunda-feira", "segundafeira", "segunda", "terça-feira", "terçafeira", "terça", "quarta-feira", "quartafeira", 
    "quarta", "quinta-feira", "quintafeira", "quinta", "sexta-feira",
    "sextafeira", "sexta", "sábado", "domingo"
]
meses_pt = [
    "janeiro",
    "fevereiro",
    "marco",
    "março",    # caso queira considerar com acento também
    "abril",
    "maio",
    "junho",
    "julho",
    "agosto",
    "setembro",
    "outubro",
    "novembro",
    "dezembro"
]
#ciclo iterrows
stops = stopwords.words("portuguese")
stops.extend(base_weekdays)

for index, row in df.iterrows():
    cleaned_data = clean_data(row['Text'],stops)

    df.at[index,'Text'] = cleaned_data

print(df)
    

       index                                               Text  \
0          0  pj operação internacional tráfico droga apreen...   
1          1  texto criado universidade república checa sist...   
2          2  nunca pague multas conduzir carta cartão cidad...   
3          3  antónio guterres estátua d afonso henriques in...   
4          4  mar droga gratuita cura menos doenças férias p...   
...      ...                                                ...   
63231  63231  governo diz estado vai indemnizar vítimas mort...   
63232  63232  homem cancro terminal realiza último desejo be...   
63233  63233  grupo impresa dívida milhões euros dívida novi...   
63234  63234  esa vai enviar sonda júpiter ciência tecnologi...   
63235  63235  mãe fica chocada perceber seguir filha superme...   

                                      Source  \
0                                         DN   
1                                        NaN   
2                            Contas Poupança   
3  

In [4]:
#Exemplo de data "limpa"
print(df.at[0,'Text'])

pj operação internacional tráfico droga apreendeu tonelada cocaína durante operação white sea iii vigiados movimentos embarcações realizadas ações busca inspeção feitas oito apreensões cocaína polícia judiciária pj participou operação internacional combate tráfico droga via marítima terminou detenção presumíveis traficantes apreensão cerca toneladas cocaína acordo comunicado divulgado feira direção nacional pj operação white sea iii decorreu junho julho durante vigiados movimentos embarcações realizadas ações busca inspeção feitas oito apreensões cocaína operação focou vários modi operandi organizações criminosas utilizam introduzir droga europa incidiu sobre movimentos embarcações determinados locais costa atlântica continente europeu canal mancha mar norte bem atividades desenvolvidas alguns portos marítimos operação incluída planos ação european multidisciplinary platform against criminal threats empcat união europeia contou participação autoridades alemanha bélgica dinamarca espanh

In [ ]:
# TF IDF podemos/devemos alterar o ngram_range e podemos colocar mais argumentos
vectorizer = TfidfVectorizer( lowercase=True,
                              max_features=5000,
                              max_df = 0.9,
                              min_df= 10,
                              ngram_range= (1,3),
                              stop_words = "english",
                              use_idf=True,
                              smooth_idf=True
                            )

vectors = vectorizer.fit_transform(df['Text'])

feature_names = vectorizer.get_feature_names_out()

dense = vectors.todense()
denselist = dense.tolist()

all_keywords = []

for description in denselist:
    x=0
    keywords = []
    for word in description:
        if word > 0:
            keywords.append(feature_names[x])
        x=x+1
    all_keywords.append(keywords)
 
print(all_keywords[0])

In [ ]:
def print_tfidf_summary(vectors):
    n_docs, n_feats = vectors.shape
    nnz = vectors.nnz
    density = nnz / (n_docs * n_feats)
    print(f"shape: {n_docs} x {n_feats}")
    print(f"non-zeros: {nnz}  density: {density:.4%}")

def top_features_for_doc(vectors, feature_names, doc_idx=0, top_n=10):
    row = vectors.getrow(doc_idx).tocoo()
    if row.nnz == 0:
        print(f"doc {doc_idx}: no non-zero features")
        return
    idx = np.argsort(row.data)[::-1][:top_n]
    terms = [feature_names[row.col[i]] for i in idx]
    scores = row.data[idx]
    print(f"Top {top_n} features for doc {doc_idx}:")
    for t, s in zip(terms, scores):
        print(f"  {t}: {s:.4f}")

def top_features_global(vectors, feature_names, top_n=30, by="mean"):
    # by="mean" or "sum"
    if by == "mean":
        vals = np.asarray(vectors.mean(axis=0)).ravel()
    else:
        vals = np.asarray(vectors.sum(axis=0)).ravel()
    top_idx = np.argsort(vals)[::-1][:top_n]
    print(f"Top {top_n} features by {by}:")
    for i in top_idx:
        print(f"  {feature_names[i]}: {vals[i]:.6f}")

def as_dataframe(vectors, feature_names, max_cells=1e9):
    # Convert only if not too large
    n_docs, n_feats = vectors.shape
    if n_docs * n_feats > max_cells:
        raise MemoryError("matrix too large to convert to dense DataFrame")
    return pd.DataFrame(vectors.toarray(), columns=feature_names)

# usage examples
print_tfidf_summary(vectors)
top_features_for_doc(vectors, feature_names, doc_idx=0, top_n=15)
top_features_global(vectors, feature_names, top_n=40, by="mean")

In [ ]:
# --- Corpus-level term summary and prints ---
# compute sums, means and document frequencies for each feature
sums = np.asarray(vectors.sum(axis=0)).ravel()
means = np.asarray(vectors.mean(axis=0)).ravel()
df_counts = np.diff(vectors.tocsc().indptr)  # documents-per-term

summary = pd.DataFrame({
    'term': feature_names,
    'sum': sums,
    'mean': means,
    'df': df_counts
})
if hasattr(vectorizer, 'idf_'):
    summary['idf'] = vectorizer.idf_

# print top terms by different metrics
def print_top_terms(summary, by='sum', top_n=30):
    print(f"\nTop {top_n} terms by {by}:")
    top = summary.sort_values(by=by, ascending=False).head(top_n)
    print(top[['term', 'sum', 'mean', 'df'] + (['idf'] if 'idf' in top.columns else [])].to_string(index=False))

print_top_terms(summary, by='sum', top_n=30)
print_top_terms(summary, by='df', top_n=30)

# show sample cleaned texts for first 5 documents
print('\nSample cleaned texts (first 5 rows):')
for i in range(min(5, len(df))):
    print(f"[{i}] ", df['Text'].iloc[i])

# save summary CSV
summary_out = 'tfidf_term_summary.csv'
summary.sort_values(by='sum', ascending=False).to_csv(summary_out, index=False)
print(f"\nSaved term summary to: {summary_out}")